# PleIAs × p7 — Constrained Generation Notebook

This notebook tests grammar-constrained decoding and the `ReasoningEnvironment`
(CoT + constrained output) with PleIAs SYNTH-series models:

| Model | Params | Vocab | Notes |
|---|---|---|---|
| `PleIAs/Monad` | 56.7M | 8 192 | English-only, edge/mobile |
| `PleIAs/Baguettotron` | 321M | 65 536 | Multilingual (EN FR DE IT ES PL) |

Both models share:
- ChatML prompt format (`<|im_start|>` / `<|im_end>`)
- Native `<think>` / `</think>` reasoning tokens
- Apache-2.0 licence, no `trust_remote_code`

**Cells 1–4** work offline (no model download).  
**Cells 5–8** require the model weights — set `MODEL_NAME` below.

## 0. Setup

In [1]:
import p7
from p7.models import PleiasConstrainedModel, get_model_class

# Choose which model to load in cells 5–8.
# Monad is much smaller — good for quick iteration on CPU.
MODEL_NAME = "PleIAs/Monad"          # 56.7M, fast
# MODEL_NAME = "PleIAs/Baguettotron"  # 321M, better quality

print(f"p7 grammars: {p7.list_grammars()}")
print(f"Adapter for {MODEL_NAME}: {get_model_class(MODEL_NAME).__name__}")

p7 grammars: ['stlc', 'imp', 'fun', 'toy']
Adapter for PleIAs/Monad: PleiasConstrainedModel


## 1. Adapter introspection (no model needed)

Inspect the class-level properties of `PleiasConstrainedModel` without loading weights.

In [2]:
print("tokenizer_kwargs :", PleiasConstrainedModel.tokenizer_kwargs())
print("model_kwargs     :", PleiasConstrainedModel.model_kwargs())

# Instance-level introspection requires a real model object.
# We can still verify the dispatch logic:
for name in ["PleIAs/Monad", "PleIAs/Baguettotron", "pleias/anything", "gpt2"]:
    cls = get_model_class(name)
    print(f"  {name:<30} -> {cls.__name__}")

tokenizer_kwargs : {'use_fast': False}
model_kwargs     : {}
  PleIAs/Monad                   -> PleiasConstrainedModel
  PleIAs/Baguettotron            -> PleiasConstrainedModel
  pleias/anything                -> PleiasConstrainedModel
  gpt2                           -> ConstrainedModel


## 2. CompletionEngine — offline constraint tests

Verify that the grammars accept the seeds we'll use in later model cells.

In [3]:
SEEDS = {
    "stlc": [
        "λx:",
        "λx:Int.",
        "λf:(Int->Bool).λx:Int.",
    ],
    "fun": [
        "(x: Int) => x",
        "let x: Int = 1; x +",
        "let double: Int -> Int = (x: Int) =>",
    ],
    "imp": [
        "{ let x: Int = 1; if (x < 5) { let y: Int = x + 1; } else { let y: Int =",
        "{ let counter: Int = 0; while (counter < 3) { counter = counter + 1; } }",
    ],
}

for grammar_name, seeds in SEEDS.items():
    print(f"\n=== {grammar_name} ===")
    for seed in seeds:
        engine = p7.CompletionEngine(p7.get_grammar(grammar_name))
        try:
            engine.feed(seed)
            completions = engine.get_completions()
            print(f"  OK  {seed!r:55s} -> {completions[:6]}")
        except Exception as exc:
            print(f"  FAIL {seed!r:55s} -> {exc}")


=== stlc ===
  OK  'λx:'                                                   -> ['(', '0']
  OK  'λx:Int.'                                               -> ['(', 'λ', 'A']
  OK  'λf:(Int->Bool).λx:Int.'                                -> ['(', 'λ', 'A']

=== fun ===
  OK  '(x: Int) => x'                                         -> ['-', '/', '(', '*', '+']
  OK  'let x: Int = 1; x +'                                   -> ['(', 'let', '0', 'a']
  OK  'let double: Int -> Int = (x: Int) =>'                  -> ['(', 'let', 'true', 'false', '0', 'a']

=== imp ===
  OK  '{ let x: Int = 1; if (x < 5) { let y: Int = x + 1; } else { let y: Int =' -> ['(', '0', 'a']
  OK  '{ let counter: Int = 0; while (counter < 3) { counter = counter + 1; } }' -> []


## 3. TypedSampler with random logits — offline

Run constrained sampling with random Gaussian logits to confirm the sampler
machinery works end-to-end before touching the real model.

In [ ]:
import random
import math

# A small but realistic char-level vocabulary
VOCAB = list(
    "abcdefghijklmnopqrstuvwxyz"
    "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    "0123456789"
    " \n"
    "λ" "->"
    ".:;,(){}[]"
    "+-*/=<>!|"
)

def random_logits():
    return [random.gauss(0, 2) for _ in VOCAB]

def sample_token(logits, temperature=0.8):
    valid = [(i, l) for i, l in enumerate(logits) if l > -1e9]
    if not valid:
        return -1
    scaled = [l / temperature for _, l in valid]
    max_l = max(scaled)
    exps = [math.exp(l - max_l) for l in scaled]
    total = sum(exps)
    probs = [e / total for e in exps]
    r = random.random()
    cumsum = 0.0
    for (idx, _), prob in zip(valid, probs):
        cumsum += prob
        if r < cumsum:
            return idx
    return valid[-1][0]

def constrained_generate_random(grammar_name, initial="", max_steps=80, seed=42):
    random.seed(seed)
    sampler = p7.TypedSampler(
        grammar=p7.get_grammar(grammar_name),
        vocab=VOCAB,
        logit_fn=random_logits,
    )
    if initial:
        sampler.feed(initial)

    for _ in range(max_steps):
        print(f"Current text: {sampler.current_text()!r}")
        masked = sampler.infer(pre_top_k=20)
        idx = sample_token(masked)
        if idx < 0:
            break
        try:
            print(f"Sampled token: {VOCAB[idx]!r} (idx={idx})")
            sampler.extend(VOCAB[idx])
        except Exception:
            break

    text = sampler.current_text()
    complete = sampler.is_complete()
    print(f"[{grammar_name}] complete={complete}  text={text!r}")
    return text, complete

print("Vocab size:", len(VOCAB))
constrained_generate_random("stlc", initial="λx:Int.")
constrained_generate_random("fun",  initial="let x: Int = 1; x +")
constrained_generate_random("imp",  initial="{ let x: Int = 1; if (x < 5) { let y: Int =")

Vocab size: 86
[stlc] complete=False  text='A'


## 4. System-prompt generation (no model needed)

`build_system_prompt` auto-generates the dual-mode CoT protocol description
for each grammar.  Inspect what Baguettotron / Monad will receive.

In [4]:
# Preview with default tags — ReasoningEnvironment will use model-specific tags
# (e.g. PleIAs uses '<think>\n' as the open token, which renders as '<think>' here).
for grammar_name in ["stlc", "fun", "imp"]:
    print(f"\n{'='*60}")
    print(f"System prompt for grammar='{grammar_name}'")
    print('='*60)
    print(p7.build_system_prompt(grammar_name))


System prompt for grammar='stlc'
You are a reasoning assistant that produces well-typed typed lambda calculus terms.

You can use two modes:
- <think>...</think>: Free-form reasoning. Think step by step.
- <stlc>...</stlc>: Produce the final well-typed output. This is grammar-constrained.

Process:
1. Use <think> to reason about the problem
2. When ready, use <stlc> to produce typed output
3. The output must be syntactically and type-correct

Syntax:
  - λx:T.e - lambda abstraction
  - (f x) - function application
  - Types use -> and are right-associative: Int->Bool->Int
  - Parenthesize function arguments and nested types when needed

Examples:
  identity: λx:Int.x
  const: λx:Int.λy:Bool.x
  apply: λf:(Int->Bool).λx:Int.(f x)

System prompt for grammar='fun'
You are a reasoning assistant that produces well-typed typed functional expressions.

You can use two modes:
- <think>...</think>: Free-form reasoning. Think step by step.
- <fun>...</fun>: Produce the final well-typed output. 

---
## 5. Load model

Requires HuggingFace weights.  Monad (~56M) fits easily on CPU;  
Baguettotron (~321M) prefers a GPU or at least 2–3 GB of RAM.

Set `MODEL_NAME` in cell 0 to switch.

In [5]:
import time

GRAMMAR_NAME = "stlc"   # grammar for cells 6–8; change as needed

print(f"Loading {MODEL_NAME} with grammar={GRAMMAR_NAME} …")
t0 = time.perf_counter()

model = PleiasConstrainedModel.from_pretrained(
    MODEL_NAME,
    grammar=p7.get_grammar(GRAMMAR_NAME),
)
model.model.eval()

elapsed = time.perf_counter() - t0
print(f"Loaded in {elapsed:.1f}s")
print(f"Vocab size : {len(model.vocab)}")
print(f"Device     : {model.device}")
print(f"think_open : {model.think_open()!r}")
print(f"think_close: {model.think_close()!r}")

# Inspect the formatted prompt
sample_prompt = model.format_prompt("Write the identity function for Int.")
print(f"\nFormatted prompt:\n{sample_prompt}")

Loading PleIAs/Monad with grammar=stlc …


/nix/store/rxnl3hc0dvvq38r5kg7997kqmszfzzs0-python3-3.12.12-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 578/578 [00:01<00:00, 529.91it/s, Materializing param=model.norm.weight]                              


Loaded in 8.8s
Vocab size : 8192
Device     : cpu
think_open : '<think>\n'
think_close: '</think>'

Formatted prompt:
<|im_start|>user
Write the identity function for Int.<|im_end>
<|im_start|>assistant



## 6. Direct constrained generation

Constrained token-by-token decoding without the ReasoningEnvironment.
Useful to isolate pure grammar constraint behaviour.

In [6]:
CONSTRAINED_TASKS = [
    {
        "grammar": "stlc",
        "prompt": "Write a simply-typed lambda term that applies f:(Int->Bool) to x:Int.",
        "initial": "λf:(Int->Bool).",
    },
    {
        "grammar": "stlc",
        "prompt": "Write the identity function for Int.",
        "initial": "λx:",
    },
    {
        "grammar": "fun",
        "prompt": "Write a typed functional expression that squares an integer.",
        "initial": "let square: Int -> Int = (x: Int) =>",
    },
]

for task in CONSTRAINED_TASKS:
    # Swap grammar if needed
    if task["grammar"] != GRAMMAR_NAME:
        model._grammar_obj = p7.Grammar(p7.get_grammar(task["grammar"]))
        model.grammar = p7.get_grammar(task["grammar"])
        GRAMMAR_NAME = task["grammar"]

    print(f"\n[{task['grammar']}] {task['prompt']}")
    print(f"  Initial: {task['initial']!r}")
    print(f"  Output : ", end="", flush=True)
    print(task["initial"], end="", flush=True)

    t0 = time.perf_counter()
    result = model.until_complete(
        prompt=task["prompt"],
        initial=task["initial"],
        max_tokens=60,
        greedy_k=3,
        pre_top_k=100,
        grammar_name=task["grammar"],
        on_token=lambda tok, _: print(tok, end="", flush=True),
    )
    elapsed = time.perf_counter() - t0

    print()
    print(f"  Complete: {result.is_complete}  |  tokens: {result.tokens_generated}  |  {elapsed*1000/max(result.tokens_generated,1):.0f}ms/tok")
    print(f"  Full text: {result.text!r}")


[stlc] Write a simply-typed lambda term that applies f:(Int->Bool) to x:Int.
  Initial: 'λf:(Int->Bool).'
  Output : λf:(Int->Bool). ((

KeyboardInterrupt: 

## 7. ReasoningEnvironment — CoT + constrained output

The model reasons freely in `<think>…</think>` blocks, then produces
type-correct output inside `<stlc>…</stlc>` blocks.

In [7]:
RE_GRAMMAR = "stlc"

# Ensure the model has the right grammar
if RE_GRAMMAR != GRAMMAR_NAME:
    model._grammar_obj = p7.Grammar(p7.get_grammar(RE_GRAMMAR))
    model.grammar = p7.get_grammar(RE_GRAMMAR)
    GRAMMAR_NAME = RE_GRAMMAR

env = p7.ReasoningEnvironment(
    model=model,
    grammar_name=RE_GRAMMAR,
    think_budget=80,
    formal_budget=40,
)

RE_TASK = {
    "prompt": "Create the K combinator: a function taking an Int and returning a function that ignores a Bool and returns the Int.",
    "initial": "λx:Int.",
}

print(f"Prompt : {RE_TASK['prompt']}")
print(f"Initial: {RE_TASK['initial']}")
print()

def on_mode_switch(mode, tag):
    print(f"\n\n[{mode.value.upper()} {tag}] ", end="", flush=True)

def on_think_token(tok, step):
    print(tok, end="", flush=True)

def on_formal_token(tok, step):
    print(f"\033[1m{tok}\033[0m", end="", flush=True)  # bold

result = env.generate(
    prompt=RE_TASK["prompt"],
    initial=RE_TASK["initial"],
    max_blocks=4,
    start_thinking=True,
    on_mode_switch=on_mode_switch,
    on_think_token=on_think_token,
    on_formal_token=on_formal_token,
)

print("\n\n" + "=" * 50)
print(f"Complete      : {result.is_complete}")
print(f"Total tokens  : {result.total_tokens}")
print(f"Think blocks  : {len(result.think_blocks)}")
print(f"Grammar blocks: {len(result.grammar_blocks)}")
if result.final_output:
    print(f"Final output  : {result.final_output.content!r}")

Prompt : Create the K combinator: a function taking an Int and returning a function that ignores a Bool and returns the Int.
Initial: λx:Int.



[THINK <think>
] 

KeyboardInterrupt: 

## 8. Batch evaluation — multiple tasks

Run the ReasoningEnvironment on a small suite of STLC tasks and summarise results.

In [ ]:
BATCH_TASKS = [
    {
        "name": "identity",
        "prompt": "Create the identity function for Int.",
        "initial": "λx:",
        "expected": "λx:Int.x",
    },
    {
        "name": "const_K",
        "prompt": "Create the K combinator: takes Int, returns a function ignoring Bool and returning the Int.",
        "initial": "λx:Int.",
        "expected": "λx:Int.λy:Bool.x",
    },
    {
        "name": "apply",
        "prompt": "Create a function applying f:(Int->Bool) to x:Int.",
        "initial": "λf:(Int->Bool).",
        "expected": "λf:(Int->Bool).λx:Int.(f x)",
    },
]

import os 
os.environ["P7_CONSTRAINED_DEBUG"] = "1"  # Enable detailed logging for debugging

batch_results = []
for task in BATCH_TASKS:
    print(f"\nTask: {task['name']}")
    t0 = time.perf_counter()
    result = env.generate(
        prompt=task["prompt"],
        initial=task["initial"],
        max_blocks=4,
        start_thinking=True,
    )
    elapsed = time.perf_counter() - t0
    output = result.final_output.content if result.final_output else None
    print(f"  Expected : {task['expected']}")
    print(f"  Got      : {output}")
    print(f"  Complete : {result.is_complete}  |  {elapsed:.1f}s  |  {result.total_tokens} tokens")
    batch_results.append({
        "name": task["name"],
        "complete": result.is_complete,
        "output": output,
        "expected": task["expected"],
        "tokens": result.total_tokens,
        "time": elapsed,
    })

print("\n" + "=" * 60)
print(f"{'Task':<12} {'Complete':<10} {'Tokens':<8} {'Time':>6}  Output")
print("-" * 60)
for r in batch_results:
    out = (r["output"] or "N/A")[:30]
    print(f"{r['name']:<12} {str(r['complete']):<10} {r['tokens']:<8} {r['time']:>5.1f}s  {out}")
complete_n = sum(1 for r in batch_results if r["complete"])
print(f"\nComplete: {complete_n}/{len(batch_results)}")


Task: identity


KeyboardInterrupt: 